In [ ]:
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch.optim as optim
import torch.nn as nn
import torch 

# Charger les données depuis le fichier CSV
# Et le nottoyage de données

chemin_vers_le_fichier = 'train.csv'

donnees = pd.read_csv(chemin_vers_le_fichier, decimal=',', sep=";", na_values=["#VALEUR!"], index_col="time")
#On remplace les cellules avec des valeurs non valides par NaN

donnees.index = pd.to_datetime(donnees.index, format='%d/%m/%Y %H:%M')
#On choisis la date en tant qu'index et on convertis au format spéciale

print(donnees.info())
#recap de la data

assert donnees.dtypes.equals(pd.Series(dict(zip(donnees.columns,["float64"]*len(donnees.columns)))))

print("\n")

donnees.isna().sum()

#Traitement des NaN on les remplace par la mediane de la collonne. La médiane est une mesure statistique qui représente la valeur centrale d'un ensemble de données triées

chosen_strategy = 'median' # "mean" / "constant" / "most_frequent"

for col in donnees.columns:
    imp_mean = SimpleImputer(missing_values=np.nan, strategy='median')
    donnees[col] = imp_mean.fit_transform(donnees[[col]]).squeeze()

assert (donnees.isna().sum(axis=0).sum() == 0)

print("\n")

donnees.isna().sum()
#Plus de valeur NaN chouette !!!! hihihihihihihihihihihihihihi

donnees = donnees.sample(n=100236, random_state=12).copy()

print(donnees.info())


# Calculer la taille des deux parties x1 et x2
taille_x1 = int(3 * len(donnees) / 4)
taille_x2 = len(donnees) - taille_x1

print(len(donnees),taille_x1,taille_x2)
# Diviser le DataFrame en deux parties x1 et x2
x1 = donnees.iloc[:taille_x1]
x2 = donnees.iloc[taille_x1:]

donnees = x1

donnees_test = x2

print(donnees.info())

# Vérifier les noms des colonnes pour les index x et y
# Remplacez 'nom_colonne_x' et 'nom_colonne_y' par les noms réels des colonnes que vous voulez utiliser

for col in donnees.columns :
    print(col)
    index_x = donnees[col]
    index_y = donnees['Net Power (MW)']

    print(index_x,index_y)

    
    plt.scatter(index_x, index_y, color='black', marker='.')
    
    plt.xlabel('Axe des Nets Powers')
    plt.ylabel(f'Axe des {col}')
    plt.title(f'Graphique des {col}')
    plt.grid(True)
    plt.show()


# Récupérer la colonne "Net Power (MW)" dans un nouveau DataFrame
donnees_labels = donnees['Net Power (MW)']

predict_label = donnees_test['Net Power (MW)']

# Supprimer la colonne "Net Power (MW)" du DataFrame initial
columns_to_drop = ["amb pressure", "Network Frequency (Hz)", "Lower Heating Value (Wh/Nm3)","Net Power (MW)"]

donnees.drop(columns=columns_to_drop, axis=1, inplace=True)

donnees_test.drop(columns=columns_to_drop, axis=1, inplace=True)

#print(donnees.info(),"\n",donnees_labels.info())


losses = []

# Convertir les données d'entraînement et les étiquettes en tenseurs
train_tensor = torch.tensor(donnees.values, dtype=torch.float32)
train_label_tensor = torch.tensor(donnees_labels.values, dtype=torch.float32)

# Convertir les données de prédiction en tenseur
predict_tensor = torch.tensor(donnees_test.values, dtype=torch.float32)

# Définir le modèle du régresseur simple
class SimpleRegressor(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRegressor, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

input_size = donnees.shape[1]
hidden_size = 16
output_size = 1

model = SimpleRegressor(input_size, hidden_size, output_size)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.2)

# Mettre le modèle en mode d'entraînement
model.train()

# Boucle d'entraînement
num_epochs = 200000  # Augmenter le nombre d'époques pour de meilleures performances

for epoch in range(num_epochs):
    # Passer les données dans le modèle
    outputs = model(train_tensor)

    # Calculer la perte
    loss = criterion(outputs.squeeze(), train_label_tensor)

    # Réinitialiser les gradients, effectuer la rétropropagation et mettre à jour les poids
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Afficher la perte à chaque époque
    #print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")
    losses.append(loss.item())

model_params = model.state_dict()

print(model_params, "\n")

plt.plot(range(1, num_epochs+1), losses, label='Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Fonction de perte en fonction du temps (nombre d\'époques)')
plt.legend()
plt.grid(True)
plt.show()

# Charger les données depuis le fichier CSV
# Et le nottoyage de données

chemin_vers_le_fichier = 'test.csv'

test = pd.read_csv(chemin_vers_le_fichier, decimal=',', sep=";", na_values=["#VALEUR!"], index_col="time")
#On remplace les cellules avec des valeurs non valides par NaN

print(test.info())

columns_to_drop = ["amb pressure", "Network Frequency (Hz)", "Lower Heating Value (Wh/Nm3)","Net Power (MW)"]

X_test = test.drop(columns=columns_to_drop, axis=1, inplace=True)


test.index = pd.to_datetime(test.index, format='%d/%m/%Y %H:%M')
#On choisis la date en tant qu'index et on convertis au format spéciale

#recap de la data

assert test.dtypes.equals(pd.Series(dict(zip(test.columns,["float64"]*len(test.columns)))))

print("\n")

test.isna().sum()

#Traitement des NaN on les remplace par la mediane de la collonne. La médiane est une mesure statistique qui représente la valeur centrale d'un ensemble de données triées

chosen_strategy = 'median' # "mean" / "constant" / "most_frequent"

for col in test.columns:
    imp_mean = SimpleImputer(missing_values=np.nan, strategy='median')
    test[col] = imp_mean.fit_transform(test[[col]]).squeeze()

assert (test.isna().sum(axis=0).sum() == 0)

print("\n")

test.isna().sum()

print(test)


# Convertir les données d'entraînement en tenseurs
train_tensor = torch.tensor(test.values, dtype=torch.float32)



# Passer les nouvelles données dans le modèle pour obtenir les prédictions
with torch.no_grad():
    predictions = model(train_tensor)

predictions = predictions.numpy().flatten()

df_predictions = pd.DataFrame({
    'time': test.index,
    'Net Power (MW)': predictions,
})

df_predictions.to_csv('predictions.csv', date_format='%d/%m/%Y %H:%M',index=False, sep=';')
df_predictions.head()
